In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import librosa.display
import logging

from src.helper import *

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
OUTPUT_PATH = "output/"

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

In [4]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_01_feature_engineering_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting feature engineering...")

### Define feature engineering function

In [5]:
def audio_feature_engineering(filepath, logger):
    features = [
        "duration",
        "sample_rate",
        "rms_energy",
        "zero_crossing_rate",
        "spectral_centroid",
    ]

    try:
        y, sr = librosa.load(filepath, sr=None)  # keep original sample rate
        duration = librosa.get_duration(y=y, sr=sr)
        
        rms = float(np.mean(librosa.feature.rms(y=y)))
        zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        
        return {
            "duration": duration,
            "sample_rate": sr,
            "rms_energy": rms,
            "zero_crossing_rate": zcr,
            "spectral_centroid": centroid
        }
    except Exception as e:
        logger.info(f"Error loading {filepath}: {e}")
        return {f:None for f in features}

### Apply feature engineering on train/test/holdout data

In [6]:
records = []

# for split in ["holdout", "testing", "training"]:
for split in ["holdout"]:
    for label in ["", "real", "fake"]:
        folder = os.path.join(DATA_PATH, split, label)
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            if os.path.isfile(fpath):
                features = audio_feature_engineering(fpath, logger)
                record = {
                    "filepath": fpath,
                    "filename": fname,
                    "split": split,
                    "label": label,
                }
                record.update(features)
                records.append(record)

df = pd.DataFrame(records)
print(df.shape)
print(df.groupby(["split", "label"]).size())
df.head(3)

/tmp/ipykernel_44906/3410206258.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filepath, sr=None)  # keep original sample rate
/home/ardacandra/miniconda3/envs/deepdetect_audio_deepfake_detection_challenge/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


(14397, 9)
split    label
holdout           14397
dtype: int64


,filepath,filename,split,label,duration,sample_rate,rms_energy,zero_crossing_rate,spectral_centroid
0,data/deep-detect/dataset/holdout/audio_06718.wav,audio_06718.wav,holdout,,2.952,16000.0,0.041040,0.165160,1770.421519
1,data/deep-detect/dataset/holdout/audio_00530.wav,audio_00530.wav,holdout,,5.742,16000.0,0.008138,0.127596,1722.722619
2,data/deep-detect/dataset/holdout/audio_12760.wav,audio_12760.wav,holdout,,1.431,16000.0,0.025449,0.125467,1871.679940


In [10]:
df.to_csv(os.path.join(OUTPUT_PATH, "nb_01__train_test_holdout_feats.csv"), index=False)
logger.info(f"output features saved to {OUTPUT_PATH}nb_01__train_test_holdout_feats.csv")